# 03. 계약서 → 엔진 팩토리 → NPV

계약서(무엇)와 평가 방법(어떻게)을 나눈다(원칙 6). 계약서는 순수 데이터이고, 엔진은 팩토리의 빌더에서 받는다.

| 단계 | QuantTool 코드 | ORE 원본 (`references/Engine/OREData/ored/portfolio/`) |
|---|---|---|
| 계약서 | `positions/ust_book.yaml`, `data/portfolio/bond.py` `Bond`, `BondData` | `bond.hpp`, `trade.hpp`, `portfolio.xml` |
| 레그 | `data/portfolio/legdata.py` `LegData`, `legbuilders.py` `FixedLegBuilder` | `legdata.hpp`, `legbuilders.hpp` |
| 엔진 선택 | `configs/pricing/engines.yaml` → `EngineData` | `pricingengine.xml`, `enginedata.hpp` |
| 팩토리·빌더 | `data/portfolio/enginefactory.py`, `builders/bond.py` | `enginefactory.hpp`, `builders/bond.hpp` |

> 데이터: `tests/fixtures/market_ust_sample.txt`는 ORE `market.txt` 형식(`날짜 키 값`)의 **합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE = ROOT / "tests" / "fixtures" / "market_ust_sample.txt"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.data.configuration import load_config
from quanttool.data.marketdata import CSVLoader, TodaysMarket
from quanttool.data.portfolio import EngineFactory, MarketContext, Portfolio

cfg = load_config(ROOT / "configs")
loader = CSVLoader(SAMPLE)
market = TodaysMarket.from_config(date(2026, 10, 23), cfg, loader)

## 1. 계약서는 데이터다
ORE `BondData`: 발행자·증권 id·**기준 커브 이름**(`reference_curve_id: UST`)·결제일·달력·발행일 + 쿠폰 레그(`LegData`). 엔진도 커브 객체도 없다. 매도 포지션은 쿠폰 레그 `payer: true`.

In [ ]:
book = Portfolio.from_file(ROOT / "positions" / "ust_book.yaml")
t = book.get("UST_10Y")
print(t.trade_type, t.bond_data.security_id, t.bond_data.reference_curve_id)
leg = t.bond_data.coupons[0]
print(leg.leg_type, "payer" if leg.payer else "receiver", leg.notionals, leg.fixed_leg_data.rates)
leg.schedule_data.rules[0].model_dump()

## 2. 엔진 팩토리와 빌더
`engines.yaml`의 `Bond: {model: DiscountedCashflows, engine: DiscountingRiskyBondEngine}`에 맞는 빌더를 팩토리가 찾아 시장·configuration·파라미터를 넣어 준다. 빌더는 **키**(통화, 신용 커브, 증권 id, 기준 커브, income 커브)가 같으면 엔진을 재사용한다.

In [ ]:
factory = EngineFactory.from_config(cfg, market)  # = EngineFactory(cfg.engine_data, market)
builder = factory.builder("Bond")
print(
    type(builder).__name__,
    builder.model,
    builder.engine_name,
    builder.configuration(MarketContext.pricing),
)
e1 = builder.engine("USD", "", "UST_10Y", "UST", "")
print("같은 키 → 같은 엔진:", builder.engine("USD", "", "UST_10Y", "UST", "") is e1)
print("레그 빌더:", type(factory.leg_builder("Fixed")).__name__)

## 3. 조립: `trade.build(factory)` (ORE `Bond::build` 순서)
1. `factory.builder('Bond')`
2. 쿠폰 레그마다 `factory.leg_builder(leg_type).build_leg(...)` → QuantLib 현금흐름
3. `ql.Bond(결제일수, 달력, 발행일, 현금흐름)` + `builder.engine(키…)`
4. `VanillaInstrument(상품, 승수)`: 승수 = bond_notional × (payer면 −1)

In [ ]:
with market.active():
    book.build(factory)
    t = book.get("UST_10Y")
    print("NPV 통화", t.npv_currency(), " 원금", t.notional(), " 만기", t.maturity())
    display(
        pd.DataFrame(
            [(cf.date().ISO(), cf.amount()) for cf in t.legs()[0]], columns=["date", "amount"]
        ).tail(3)
    )
    display(book.npvs().round(2))
    print("합계", round(book.npv(), 2))

## 4. QuantLib으로 직접 계산한 값과 같은가

In [ ]:
with market.active():
    sch = ql.Schedule(
        ql.Date(15, 8, 2026),
        ql.Date(15, 8, 2036),
        ql.Period(ql.Semiannual),
        ql.UnitedStates(ql.UnitedStates.GovernmentBond),
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    direct = ql.FixedRateBond(
        1,
        1.0e7,
        sch,
        [0.04125],
        ql.ActualActual(ql.ActualActual.Bond),
        ql.Unadjusted,
        100.0,
        ql.Date(15, 8, 2026),
    )
    direct.setPricingEngine(ql.DiscountingBondEngine(market.yield_curve("UST")))
    print(direct.NPV(), book.get("UST_10Y").npv())

## 5. 같은 계약서, 다른 날짜의 시장
계약서는 그대로 두고 다른 시장으로 다시 조립한다. 평가는 각 시장의 `active()` 안에서.

In [ ]:
prev_market = TodaysMarket.from_config(date(2026, 10, 22), cfg, loader)
with prev_market.active():
    book.build(EngineFactory.from_config(cfg, prev_market))
    prev = book.npvs()
with market.active():
    book.build(EngineFactory.from_config(cfg, market))
    now = book.npvs()
pd.DataFrame({"10/22": prev, "10/23": now, "손익": now - prev}).round(2)

다음: `04_simmarket_sensitivity.ipynb` — 시뮬레이션 시장을 흔들어 민감도를 본다.